In [1]:
import json
import os
import shutil

In [11]:
def process_node(node):
    if 'raw_text' in node:
        return node['raw_text']

    if 'AND' in node:
        left_text = process_node(node['AND']['left'])
        right_text = process_node(node['AND']['right'])
        return f"{left_text} [AND]\n{right_text}"

    if 'OR' in node:
        left_text = process_node(node['OR']['left'])
        right_text = process_node(node['OR']['right'])
        return f"{left_text} [OR]\n{right_text}"

    if 'NOT' in node:
        left_text = process_node(node['NOT']['left'])
        return f"[NOT] {left_text}\n"

    return ""

def json_to_text(json_file_path):
    with open(json_file_path, 'r') as f:
        data = json.load(f)

    text = process_node(data)
    return text

def process_all_files_in_directory(directory_path, output_directory, failure_directory):
    if not os.path.exists(output_directory):
        os.makedirs(output_directory)

    if not os.path.exists(failure_directory):
        os.makedirs(failure_directory)

    for filename in os.listdir(directory_path):
        if filename.endswith("_exc.json") or filename.endswith("_inc.json"):
            json_file_path = os.path.join(directory_path, filename)
            try:
                output_text = json_to_text(json_file_path)
                output_file_path = os.path.join(output_directory, os.path.splitext(filename)[0] + ".txt")
                with open(output_file_path, 'w') as output_file:
                    output_file.write(output_text)
                print(f"Processed file: {filename}")
            except Exception as e:
                print(f"Failed to process file: {filename} - Error: {e}")
                shutil.copy(json_file_path, os.path.join(failure_directory, filename))




### Json Output zu Txt Files mit Operatoren

In [12]:
# Beispielnutzung
model = "Llama3_70b_Fine-Tuned_p3_ep5_p1"
model_name = f"model_output/{model}"  # Ersetzen Sie dies durch den tatsächlichen Modellnamen
input_directory_path = os.path.join(model_name, "output")
output_directory_path = os.path.join(model_name, "processed_output")
failure_directory_path = os.path.join(model_name, "failure")
process_all_files_in_directory(input_directory_path, output_directory_path, failure_directory_path)

Failed to process file: Llama3_70b_Fine-Tuned_p3_ep5_NCT03860220_inc.json - Error: 'charmap' codec can't encode character '\u2264' in position 11: character maps to <undefined>
Processed file: Llama3_70b_Fine-Tuned_p3_ep5_NCT03860259_inc.json
Processed file: Llama3_70b_Fine-Tuned_p3_ep5_NCT03860311_exc.json
Processed file: Llama3_70b_Fine-Tuned_p3_ep5_NCT03860324_exc.json
Failed to process file: Llama3_70b_Fine-Tuned_p3_ep5_NCT03860363_inc.json - Error: 'charmap' codec can't encode character '\u2265' in position 13: character maps to <undefined>
Processed file: Llama3_70b_Fine-Tuned_p3_ep5_NCT03860402_exc.json
Processed file: Llama3_70b_Fine-Tuned_p3_ep5_NCT03860402_inc.json
Processed file: Llama3_70b_Fine-Tuned_p3_ep5_NCT03860428_inc.json
Processed file: Llama3_70b_Fine-Tuned_p3_ep5_NCT03860493_exc.json
Processed file: Llama3_70b_Fine-Tuned_p3_ep5_NCT03860545_inc.json
Processed file: Llama3_70b_Fine-Tuned_p3_ep5_NCT03860597_inc.json
Failed to process file: Llama3_70b_Fine-Tuned_p3_ep5

In [13]:
ic = "Inclusion Criteria:"
ec = "Exclusion Criteria:"

In [14]:
def combine_files(input_dir, output_dir):
    def read_file(filepath):
        with open(filepath, 'r', encoding="utf-8") as file:
            return file.read()

    def write_combined_file(nct_number, inc_text, exc_text, output_dir):
        output_filepath = os.path.join(output_dir, f'NCT{nct_number}.txt')
        with open(output_filepath, 'w', encoding="utf-8") as file:
            file.write(f"Inclusion Criteria:\n{inc_text}\n")
            file.write(f"Exclusion Criteria:\n{exc_text}\n")

    if not os.path.exists(output_dir):
        os.makedirs(output_dir)

    files = os.listdir(input_dir)
    nct_numbers = set()

    for file in files:
        if 'inc' in file or 'exc' in file:
            nct_number = file.split('_')[1]
            nct_numbers.add(nct_number)
    
    for nct_number in nct_numbers:
        inc_files = [f for f in files if nct_number in f and 'inc' in f]
        exc_files = [f for f in files if nct_number in f and 'exc' in f]
    
        if inc_files and exc_files:
            inc_text = '\n'.join(read_file(os.path.join(input_dir, f)) for f in inc_files)
            exc_text = '\n'.join(read_file(os.path.join(input_dir, f)) for f in exc_files)
            print(inc_text)
            print(exc_text)
            write_combined_file(nct_number, inc_text, exc_text, output_dir)

In [15]:
input_dir = f'model_output/{model}/processed_output'
output_dir = f'model_output/{model}/combined_output'

combine_files(input_dir, output_dir)

UnicodeDecodeError: 'utf-8' codec can't decode byte 0xb0 in position 131: invalid start byte